In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)


In [9]:
from pyspark.sql import SparkSession
import requests
import time
import json
from datetime import datetime


ModuleNotFoundError: No module named 'schedule'

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("[INFO] Sessão Spark iniciada com sucesso!")

[INFO] Sessão Spark iniciada com sucesso!


In [4]:
spark.createDataFrame([{"teste": 1}]).write.mode("overwrite").json("s3a://bronze/teste/")

In [5]:
def coletar_dados():
    API_TOKEN = "84dfde66637e13d4307f9408fc4d5c36474c7dc322310ef71f264dda3fe75704"
    LOGIN_URL = f"https://api.olhovivo.sptrans.com.br/v2.1/Login/Autenticar?token={API_TOKEN}"

    print("[INFO] Fazendo login na API SPTrans...")
    login_response = requests.post(LOGIN_URL)

    if login_response.status_code != 200:
        raise Exception(f"[ERRO] Falha no login: {login_response.status_code}")

    cookie_value = login_response.cookies.get("apiCredentials")
    if not cookie_value:
        raise Exception("[ERRO] Cookie não retornado pela API")

    print("[INFO] Login bem-sucedido! Capturado cookie de sessão.")

    # Linhas a serem coletadas
    LINHAS = [544, 198, 1989]
    registros = []
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for codigo in LINHAS:
        try:
            url = f"https://api.olhovivo.sptrans.com.br/v2.1/Previsao/Linha?codigoLinha={codigo}"
            resp = requests.get(url, cookies={"apiCredentials": cookie_value})

            if resp.status_code == 200:
                registros.append({
                    "codigo_linha": codigo,
                    "timestamp": ts,
                    "raw_json": resp.text
                })
                print(f"[OK] Linha {codigo} coletada com sucesso.")
            else:
                registros.append({
                    "codigo_linha": codigo,
                    "timestamp": ts,
                    "raw_json": f"ERRO {resp.status_code}"
                })
                print(f"[ERRO] Linha {codigo}: {resp.status_code}")

        except Exception as e:
            print(f"[ERRO] Falha ao coletar linha {codigo}: {e}")
            registros.append({
                "codigo_linha": codigo,
                "timestamp": ts,
                "raw_json": f"ERRO: {e}"
            })

    return registros

In [6]:
try:
    print("[INFO] Iniciando gravação na camada Bronze...")

    dados = coletar_dados()              # chama a função de coleta
    df = spark.createDataFrame(dados)    # cria DataFrame Spark

    output_path = "s3a://bronze/sptrans/previsao/"
    (
        df.write
        .mode("append")
        .partitionBy("codigo_linha")
        .json(output_path)
    )

    print(f"[SUCESSO] {len(dados)} registros gravados em {output_path}")

except Exception as e:
    print(f"[ERRO] Falha durante execução: {e}")

finally:
    spark.stop()
    print("[INFO] Execução finalizada.")

[INFO] Iniciando gravação na camada Bronze...
[INFO] Fazendo login na API SPTrans...
[INFO] Login bem-sucedido! Capturado cookie de sessão.
[OK] Linha 544 coletada com sucesso.
[OK] Linha 198 coletada com sucesso.
[OK] Linha 1989 coletada com sucesso.
[SUCESSO] 3 registros gravados em s3a://bronze/sptrans/previsao/
[INFO] Execução finalizada.


In [21]:
try:
    print("[INFO] Iniciando coleta de dados da SPTrans...")
    dados = coletar_dados()
    df = spark.createDataFrame(dados)

    output_path = "s3a://bronze/sptrans/previsao/"
    (
        df.write
        .mode("append")
        .partitionBy("codigo_linha")
        .json(output_path)
    )

    print(f"[SUCESSO] {len(dados)} registros gravados em {output_path}")

except Exception as e:
    print(f"[ERRO] Falha durante execução: {e}")

finally:
    spark.stop()
    print("[INFO] Execução finalizada.")

[INFO] Iniciando coleta de dados da SPTrans...
[INFO] Fazendo login na API SPTrans...
[INFO] Login bem-sucedido! Capturado cookie de sessão.
[ERRO] Falha durante execução: 'NoneType' object is not iterable
[INFO] Execução finalizada.


In [10]:
pip install schedule

Note: you may need to restart the kernel to use updated packages.


In [11]:
import schedule

In [ ]:
def rodar_job():
    os.system("spark-submit sptrans_bronze_batch.py")

schedule.every(20).minutes.do(rodar_job)

print("[INFO] Job agendado a cada 20 minutos. Pressione Ctrl+C para parar.")
while True:
    schedule.run_pending()
    time.sleep(60)

[INFO] Job agendado a cada 20 minutos. Pressione Ctrl+C para parar.
